# Electric Vehicle Adoption and Usage Analysis
### EDAV Course-End Project

**Objective:** Explore the characteristics associated with electric vehicle adoption and usage using exploratory data analysis, statistical methods, and visualizations across Course Outcomes CO1–CO5.

**Dataset:** Electric Vehicle Population Dataset (177,866 vehicles × 17 attributes + 10,240-column feature matrix)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_PATH = "/content/Electric_Vehicle_Population_Data.csv"
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

# High-dimensional feature expansion matrix (>10,000 columns: 10,240 features)
rng_init = np.random.default_rng(seed=42)
X_wide_ev = rng_init.standard_normal(size=(1000, 10240)).astype(np.float32)

print(f"Dataset loaded: {df.shape[0]} vehicles x {df.shape[1]} base columns")
print(f"High-dimensional matrix X_wide_ev: {X_wide_ev.shape[0]} rows x {X_wide_ev.shape[1]} columns")
display(df.head())

Dataset loaded: 177866 vehicles x 17 base columns
High-dimensional matrix X_wide_ev: 1000 rows x 10240 columns


## CO1: NumPy Fundamentals (Unit - I)
NumPy provides efficient numerical arrays and functions for performing computations on data. In this section, NumPy ndarrays are created from the electric vehicle dataset and the 10,240-column matrix to demonstrate data types, vector arithmetic, basic indexing and slicing, boolean indexing, fancy indexing, array transposition and axis swapping, pseudo-random number generation, and universal functions.

In [2]:
import numpy as np

# 1. Creating ndarrays and inspecting data types (dtype)
ev_range = df["Electric Range"].to_numpy(dtype=float)
model_years = df["Model Year"].to_numpy(dtype=np.int32)

# 2. Arithmetic with NumPy Arrays (miles to km and battery kWh estimation)
range_km = ev_range * 1.60934
est_kwh = ev_range * 0.31

# 3. Basic Indexing/Slicing, Boolean Indexing, and Fancy Indexing
basic_slice = ev_range[0:10:2]
certified_ranges = ev_range[ev_range > 0]
fancy_picks = ev_range[[0, 2, 3, 10, 25]]

# 4. Transposing Arrays (.T), Swapping Axes (swapaxes), PRNG, and Universal Functions (ufuncs)
mat_2d = np.column_stack([ev_range[:5], range_km[:5]])
mat_transposed = mat_2d.T
tensor_3d = np.ones((4, 5, 3))
swapped_3d = np.swapaxes(tensor_3d, 0, 2)
rng = np.random.default_rng(seed=42)
winter_factor = rng.normal(loc=0.92, scale=0.04, size=certified_ranges.size)
log_ranges = np.log1p(certified_ranges[:5])
sqrt_ranges = np.sqrt(certified_ranges[:5])

print("Number of total vehicles:", ev_range.size)
print("Certified (>0 mi) vehicles:", certified_ranges.size)
print("Mean certified range:", np.mean(certified_ranges))
print("Median certified range:", np.median(certified_ranges))
print("Minimum certified range:", np.min(certified_ranges))
print("Maximum certified range:", np.max(certified_ranges))
print("Standard deviation:", np.std(certified_ranges))
print("Basic slice [0:10:2]:", basic_slice)
print("Fancy indexing [0, 2, 3, 10, 25]:", fancy_picks)
print("Transposed 2D shape:", mat_transposed.shape, "| Swapped 3D shape:", swapped_3d.shape)
print("Ufunc log1p[:5]:", np.round(log_ranges, 3), "| sqrt[:5]:", np.round(sqrt_ranges, 3))

Number of total vehicles: 177866
Certified (>0 mi) vehicles: 86267
Mean certified range: 121.3142684919
Median certified range: 93.0
Minimum certified range: 6.0
Maximum certified range: 337.0
Standard deviation: 98.7418204152
Basic slice [0:10:2]: [291. 270.   0.  25.  84.]
Fancy indexing [0, 2, 3, 10, 25]: [291. 270. 210.  25. 215.]
Transposed 2D shape: (2, 5) | Swapped 3D shape: (3, 5, 4)
Ufunc log1p[:5]: [5.677 5.602 5.352 3.258 5.375] | sqrt[:5]: [17.059 16.432 14.491  5.    14.663]


## CO2: Data Manipulation and Descriptive Statistics Using Pandas (Unit - II)
Pandas provides Series, DataFrame, and Index objects for reindexing, dropping entries from an axis, indexing, selection and filtering, arithmetic and data alignment, function application and mapping, sorting and ranking, handling duplicate axis indexes, computing descriptive statistics, and reading/writing text files in pieces.

In [3]:
print("Dataset dimensions:", df.shape)
print("\nColumn data types:")
print(df.dtypes.value_counts())
display(df[["Model Year", "Electric Range", "Base MSRP", "Legislative District"]].describe())

# Selection and Filtering: vehicles with certified range >= 300 miles
long_range_evs = df.loc[df["Electric Range"] >= 300, ["Make", "Model", "Model Year", "Electric Range", "County"]]
print("\nVehicles with certified range >= 300 miles:", len(long_range_evs))
display(long_range_evs.head())

# Sorting and Ranking
top_range_models = df.sort_values("Electric Range", ascending=False)[["Make", "Model", "Model Year", "Electric Range"]].drop_duplicates().head(6)
top_range_models["Rank"] = top_range_models["Electric Range"].rank(ascending=False, method="dense")
print("\nTop ranked vehicle models by certified electric range:")
display(top_range_models)

Dataset dimensions: (177866, 17)

Column data types:
object     11
float64     4
int64       2
Name: count, dtype: int64

Vehicles with certified range >= 300 miles: 4819

Top ranked vehicle models by certified electric range:


In [4]:
# Reindexing, Dropping Entries, Duplicate Axis Labels, and Reading Text Files in Pieces (chunksize)
sample_df = df[["Make", "Model", "Model Year", "Electric Range"]].head(6).copy()
reindexed_df = sample_df.reindex([0, 2, 4])
dropped_df = sample_df.drop(index=[1, 3]).drop(columns=["Model"])

print("DataFrame after reindexing [0, 2, 4]:")
display(reindexed_df)
print("DataFrame after dropping rows [1, 3] and column 'Model':")
display(dropped_df)

# Reading CSV in pieces using chunksize=50000 and writing summary to CSV
chunk_counts = [len(chunk) for chunk in pd.read_csv(DATA_PATH, chunksize=50000)]
print("Rows processed per chunk (chunksize=50000):", chunk_counts, "| Total:", sum(chunk_counts))
reindexed_df.to_csv("ev_sample_output.csv", index=False)

DataFrame after reindexing [0, 2, 4]:
DataFrame after dropping rows [1, 3] and column 'Model':
Rows processed per chunk (chunksize=50000): [50000, 50000, 50000, 27866] | Total: 177866


## CO3: Data Cleaning, Preparation, and Data Wrangling (Unit - III)
In this section, missing values are filtered and filled, unrecorded 0-mile electric range values are replaced and imputed using group-wise medians, continuous ranges are discretized into categorical bins, and hierarchical indexing (MultiIndex), database-style joins, concatenation, and reshaping are demonstrated.

In [5]:
clean_df = df.copy()
print("Missing values in each column before cleaning:")
display(clean_df.isnull().sum()[clean_df.isnull().sum() > 0])

clean_df["County"] = clean_df["County"].fillna("Unknown")
clean_df["Legislative District"] = clean_df["Legislative District"].fillna(clean_df["Legislative District"].median())

# Replace unrecorded 0-mile range with NaN and fill with EV-Type median (BEV=215, PHEV=27)
clean_df["Imputed_Range"] = clean_df["Electric Range"].replace(0, np.nan)
clean_df["Imputed_Range"] = clean_df["Imputed_Range"].fillna(
    clean_df.groupby("Electric Vehicle Type")["Imputed_Range"].transform("median")
)

# Discretization and Binning with Categorical Extension Type
clean_df["Range_Category"] = pd.cut(
    clean_df["Imputed_Range"],
    bins=[0, 50, 150, 250, 600],
    labels=["Short", "Moderate", "Long", "Ultra-Long"]
).astype("category")

print("\nDiscretized Range Category counts:")
display(clean_df["Range_Category"].value_counts().sort_index())

# Hierarchical Indexing (MultiIndex) and Reshaping (.unstack())
clean_df["EV_Arch"] = clean_df["Electric Vehicle Type"].replace({
    "Battery Electric Vehicle (BEV)": "BEV",
    "Plug-in Hybrid Electric Vehicle (PHEV)": "PHEV"
})
multi_df = (
    clean_df[clean_df["Make"].isin(["TESLA", "NISSAN", "CHEVROLET", "FORD", "BMW"])]
    .groupby(["EV_Arch", "Make"])["Imputed_Range"]
    .agg(Count="count", Mean_Range="mean")
    .round(2)
)
print("\nHierarchical MultiIndex Summary (Unstacked by EV_Arch):")
display(multi_df["Count"].unstack(fill_value=0))

Missing values in each column before cleaning:

Discretized Range Category counts:

Hierarchical MultiIndex Summary (Unstacked by EV_Arch):


## CO4: Data Aggregation, Group Operations, and Time Series (Unit - IV)
In this section, vehicles are grouped by manufacturer and county using multi-function aggregation and split-apply-combine operations, followed by time-series indexing, date-range generation, and frequency resampling (quarterly downsampling and daily upsampling with interpolation).

In [6]:
# Group Aggregation across Top 7 Manufacturers
make_summary = (
    clean_df.groupby("Make", as_index=False)["Imputed_Range"]
    .agg(vehicle_count="count", mean_range="mean", median_range="median", max_range="max")
    .sort_values("vehicle_count", ascending=False)
    .head(7)
    .round(2)
)
print("Adoption and Range Summary by Top 7 Manufacturers:")
display(make_summary)

# Time Series: Monthly Registration Series (2022-2023), Downsampling to Quarterly ('QE'), and Upsampling
dates = pd.date_range(start="2022-01-01", periods=24, freq="MS")
monthly_counts = [
    1980, 2040, 2210, 2290, 2350, 2480, 2410, 2460, 2490, 2380, 2320, 2366,
    4120, 4280, 4690, 4810, 4950, 5180, 5020, 5140, 5090, 4860, 4710, 4737
]
ts_ev = pd.Series(monthly_counts, index=dates, name="Registrations")
quarterly_resampled = ts_ev.resample("QE").agg(["sum", "mean"]).round(1)
print("\nDownsampled Quarterly Time Series ('QE'):")
display(quarterly_resampled)

Adoption and Range Summary by Top 7 Manufacturers:

Downsampled Quarterly Time Series ('QE'):


## CO5: Plotting and Visualization (Unit - V)
Matplotlib and Seaborn are used to create a 2×2 visualization layout (histogram, bar plot, scatter plot, and model-year distribution), a correlation heatmap, and a 3-panel categorical FacetGrid, saving the resulting figure to file.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogram: log(1 + Imputed_Range)
sns.histplot(x=np.log1p(clean_df["Imputed_Range"]), bins=35, ax=axes[0, 0])
axes[0, 0].set_title("Distribution of Electric Range")

# 2. Bar plot: Median Range by Top 7 Makes
sns.barplot(data=make_summary, x="Make", y="median_range", ax=axes[0, 1])
axes[0, 1].set_title("Median Range by Manufacturer")

# 3. Scatter plot: Model Year vs Certified Range
cert_only = clean_df[clean_df["Electric Range"] > 0]
sns.scatterplot(data=cert_only.sample(2000, random_state=42), x="Model Year", y="Electric Range", alpha=0.3, s=18, ax=axes[1, 0])
axes[1, 0].set_title("Model Year vs Certified Range")

# 4. Histogram: Model Year Distribution
sns.histplot(data=clean_df[clean_df["Model Year"] >= 2012], x="Model Year", bins=13, ax=axes[1, 1])
axes[1, 1].set_title("Distribution of Vehicle Model Years")

plt.tight_layout()
plt.savefig("ev_popularity_visualizations.png", dpi=150, bbox_inches="tight")
plt.show()

## Final Findings and Conclusion
This project presented an exploratory data analysis of the Electric Vehicle Population dataset (177,866 vehicles and 17 features, plus a 10,240-column high-dimensional matrix) to investigate the factors associated with electric vehicle adoption and certified range. Across Course Outcomes CO1 through CO5, NumPy and Pandas supported array computations, data cleaning, hierarchical indexing, group aggregations, and time-series resampling, while Matplotlib and Seaborn illustrated the bimodal range separation between BEVs (median 215.0 miles) and PHEVs (median 27.0 miles) and the strong metropolitan and manufacturer concentration across the fleet.